[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C52_Industrial_Research_Practice_Course/01_tensorflow/01_tensorflow_mental_model.ipynb)

# 01 · TensorFlow / Keras 心智模型（用 numpy 复现 tf.function 的追踪语义）

目标：把 **静态图追踪 → Python 只跑一次 → 控制流固化 → 重追踪与缓存 → 与 PyTorch 的逐概念对照**
用一个几十行的 tracing 编译器精确复现。

路线：迷你 tracer 与图 → **print/计数器只在追踪时执行** → Python if 被固化 →
tf.cond 式的图内控制流 → 签名缓存与重追踪 → **重追踪诊断技巧** →
框架默认值对照表（三个「不报错但数值错」的坑）→ 编译的盈亏平衡 → ✏️ 练习 → 📖 答案 → 🧪 对照胶囊。

> 心智模型：**「Python 代码只在追踪时执行一次」这一句话，能解释静态图框架里 80% 的诡异行为。**

## 1 · 一个迷你 tracer：Python 代码怎么变成图

追踪的核心机制极简：**用一个「符号张量」（tracer）代替真实数组去跑一遍 Python 函数**。
tracer 的每个运算不真的计算，而是**往图里加一个节点**并返回一个新的 tracer。

In [ ]:
import numpy as np, math, collections, itertools
rng = np.random.default_rng(0)

class Graph:
    def __init__(self):
        self.nodes = []          # [(op, [输入id], 属性)]
        self.inputs = []
        self._n = 0
    def new_id(self):
        self._n += 1; return f'v{self._n}'
    def add(self, op, inputs, **attrs):
        out = self.new_id()
        self.nodes.append({'op': op, 'inputs': list(inputs), 'out': out, 'attrs': attrs})
        return out
    def __repr__(self):
        return '\n'.join(f'  {n["out"]} = {n["op"]}({", ".join(n["inputs"])})'
                          + (f' {n["attrs"]}' if n['attrs'] else '')
                          for n in self.nodes)

class Tracer:
    '''符号张量：不持有数值，只持有 (图, 节点id, 形状, dtype)。'''
    def __init__(self, graph, node_id, shape, dtype='float32'):
        self.g, self.id, self.shape, self.dtype = graph, node_id, shape, dtype
    def _bin(self, other, op):
        oid = other.id if isinstance(other, Tracer) else self.g.add('const', [], value=other)
        shape = self.shape                      # 简化：假设可广播到自身形状
        return Tracer(self.g, self.g.add(op, [self.id, oid]), shape, self.dtype)
    def __add__(self, o): return self._bin(o, 'add')
    def __mul__(self, o): return self._bin(o, 'mul')
    def __matmul__(self, o):
        shape = (self.shape[0], o.shape[1])
        return Tracer(self.g, self.g.add('matmul', [self.id, o.id]), shape, self.dtype)
    def sum(self):
        return Tracer(self.g, self.g.add('sum', [self.id]), (), self.dtype)
    def __repr__(self):
        return f'Tracer(id={self.id}, shape={self.shape}, dtype={self.dtype})'
    # ⚠️ 关键：任何试图把 tracer 当成具体值用的操作都必须报错
    def __bool__(self):
        raise TypeError(
            'Tracer 不能用于 Python 的 if/while/bool()：追踪时它没有具体值。\n'
            '  -> 依赖运行时值的控制流必须用图算子表达（tf.cond / lax.cond / torch.cond）')
    def __int__(self):
        raise TypeError('Tracer 不能转成 int：追踪时它没有具体值')

def trace(fn, input_specs):
    '''用 tracer 跑一遍 fn，得到一张图。input_specs: [(shape, dtype)]'''
    g = Graph()
    args = []
    for shape, dtype in input_specs:
        nid = g.new_id(); g.inputs.append(nid)
        g.nodes.append({'op': 'placeholder', 'inputs': [], 'out': nid,
                        'attrs': {'shape': shape, 'dtype': dtype}})
        args.append(Tracer(g, nid, shape, dtype))
    out = fn(*args)
    return g, out

def f(x, w):
    h = x @ w
    return h + 1.0

g, out = trace(f, [((4, 8), 'float32'), ((8, 8), 'float32')])
print('追踪得到的图:'); print(g)
print('输出:', out)
assert len(g.nodes) == 5, '2 个 placeholder + matmul + const(1.0) + add'
assert any(n['op'] == 'matmul' for n in g.nodes)
print('\n✅ 追踪 = 用符号张量跑一遍 Python，把每个算子记成图节点。')
print('   **注意 Tracer.__bool__ 直接抛错** —— 这正是 JAX 的做法（宁可报错也不静默固化）。')

## 2 · Python 只在追踪时执行一次：print、计数器、控制流

这是静态图最反直觉、也最能解释「诡异行为」的一条。用三个可运行的例子把它钉死。

In [ ]:
trace_log = []

def f_with_print(x, w):
    trace_log.append('Python 执行了一次')     # ← 纯 Python 副作用
    print('  [追踪期] 这行 print 现在执行')
    return (x @ w).sum()

print('第一次追踪:')
g1, _ = trace(f_with_print, [((4, 8), 'float32'), ((8, 8), 'float32')])
print('第二次追踪（同样的签名，真实框架会命中缓存、不再追踪）:')
g2, _ = trace(f_with_print, [((4, 8), 'float32'), ((8, 8), 'float32')])
print(f'\nPython 副作用执行了 {len(trace_log)} 次（我们手动追踪了 2 次）')
assert len(trace_log) == 2
print('\n⚠️  在真实的 tf.function 里，第二次调用会**命中缓存、不再执行 Python**，')
print('    所以 print 只出现一次然后「消失」—— 而调试时你会以为代码没被执行。')
print('✅ 正确做法：用 tf.print（图内算子，每次执行都打印），或临时关掉编译调试。')

In [ ]:
# Python 的 if 会被**固化**成追踪时那一条分支
def f_python_if(x, w, use_relu):
    '''use_relu 是 **Python bool** -> 分支在追踪时就定死了。'''
    h = x @ w
    if use_relu:                          # ← Python 层面的判断
        return Tracer(h.g, h.g.add('relu', [h.id]), h.shape, h.dtype)
    return h

g_relu, _ = trace(lambda x, w: f_python_if(x, w, True),
                  [((4, 8), 'float32'), ((8, 8), 'float32')])
g_norelu, _ = trace(lambda x, w: f_python_if(x, w, False),
                    [((4, 8), 'float32'), ((8, 8), 'float32')])
ops_relu = [n['op'] for n in g_relu.nodes]
ops_norelu = [n['op'] for n in g_norelu.nodes]
print('use_relu=True  的图:', ops_relu)
print('use_relu=False 的图:', ops_norelu)
assert 'relu' in ops_relu and 'relu' not in ops_norelu
print('\n⚠️  两张**不同的图** —— 分支已经被固化，运行时无法再改。')
print('    如果 use_relu 是从 tensor 算出来的，Python if 会直接报错（见下）。')

# 依赖 tensor 值的 if -> 必须报错（否则就是静默固化，更糟）
def f_tensor_if(x, w):
    h = (x @ w).sum()
    if h > 0:                             # ← 试图对 tracer 做 Python 判断
        return h
    return h * 0
try:
    trace(f_tensor_if, [((4, 8), 'float32'), ((8, 8), 'float32')])
    raise RuntimeError('不该到这')
except TypeError as e:
    print(f'\n✅ 依赖 tensor 值的 Python if 被拦下:\n   {str(e).splitlines()[0]}')

In [ ]:
# 图内控制流：把条件表达成一个图节点（tf.cond / lax.cond / torch.cond）
def graph_cond(pred, true_fn, false_fn, *args):
    '''复刻 tf.cond：**两个分支都被追踪进图**，运行时按 pred 选一个执行。'''
    g = pred.g
    t_out = true_fn(*args)
    f_out = false_fn(*args)
    return Tracer(g, g.add('cond', [pred.id, t_out.id, f_out.id]), t_out.shape, t_out.dtype)

def f_graph_if(x, w):
    h = x @ w
    s = h.sum()
    return graph_cond(s,
                      lambda: Tracer(h.g, h.g.add('relu', [h.id]), h.shape, h.dtype),
                      lambda: h)

g_cond, _ = trace(f_graph_if, [((4, 8), 'float32'), ((8, 8), 'float32')])
ops = [n['op'] for n in g_cond.nodes]
print('图内控制流的图:', ops)
assert 'cond' in ops and 'relu' in ops
print('\n✅ 一张图就够（不用为每个分支各追踪一次），且分支在**运行时**决定。')
print('   代价：**两个分支都要被追踪**（都要能通过形状检查），且图更复杂。')
print('   TF 的 AutoGraph 会自动把「条件是 tensor」的 if 改写成这种形式 ——')
print('   但「有时改写、有时不改写」正是 TF 最难懂的地方，最好显式知道每个条件是哪种。')

## 3 · 签名缓存与重追踪：性能问题的头号来源

$$\text{cache key} = (\text{dtype}_i, \text{shape}_i, \dots, \text{python\_args})$$

**签名变了就重追踪。追踪一次约等于跑几十到几百次前向。**

In [ ]:
class Function:
    '''复刻 tf.function：按签名缓存图。'''
    TRACE_COST = 200          # 追踪一次 ≈ 200 次前向的开销（量级）

    def __init__(self, fn, input_signature=None):
        self.fn, self.cache = fn, {}
        self.input_signature = input_signature
        self.n_traces = 0
        self.n_calls = 0

    def _key(self, specs, py_args):
        if self.input_signature is not None:
            specs = self.input_signature          # 声明了签名 -> key 与实际形状无关
        return (tuple(specs), tuple(sorted(py_args.items())))

    def __call__(self, specs, **py_args):
        self.n_calls += 1
        k = self._key(specs, py_args)
        if k not in self.cache:
            self.n_traces += 1
            self.cache[k] = trace(lambda *a: self.fn(*a, **py_args), specs)[0]
        return self.cache[k]

    def cost(self, t_graph=1.0, t_eager=1.3):
        return self.n_traces * self.TRACE_COST + self.n_calls * t_graph, self.n_calls * t_eager

def model(x, w, scale=1.0):
    return (x @ w) * scale

# 场景 A：形状固定
fa = Function(model)
for _ in range(500):
    fa([((32, 8), 'float32'), ((8, 8), 'float32')])
# 场景 B：形状每次都变（变长序列的典型情形）
fb = Function(model)
for i in range(500):
    fb([((32, 8 + i % 200), 'float32'), ((8 + i % 200, 8), 'float32')])
# 场景 C：形状分 8 个桶
fc = Function(model)
BUCKETS = [64, 96, 128, 192, 256, 384, 512, 768]
for i in range(500):
    b = BUCKETS[i % len(BUCKETS)]
    fc([((32, b), 'float32'), ((b, 8), 'float32')])
# 场景 D：传 Python 标量参数
fd = Function(model)
for i in range(500):
    fd([((32, 8), 'float32'), ((8, 8), 'float32')], scale=float(i % 50))

print(f"{'场景':<26s} {'调用':>6s} {'追踪':>6s} {'图缓存':>7s} {'相对 eager':>11s}")
for name, f_ in [('A 形状固定', fa), ('B 形状每次都变', fb),
                 ('C 形状分 8 桶', fc), ('D 传 Python 标量', fd)]:
    tg, te = f_.cost()
    print(f'{name:<26s} {f_.n_calls:>6d} {f_.n_traces:>6d} {len(f_.cache):>7d} {tg/te:>10.2f}×')

assert fa.n_traces == 1, '形状固定 -> 只追踪一次'
assert fb.n_traces > 100, '形状每次都变 -> 几乎每次都重追踪'
assert fc.n_traces == len(BUCKETS), '分桶 -> 追踪次数 = 桶数'
assert fd.n_traces == 50, 'Python 标量参数参与 cache key -> 每个取值一张图'
print(f'\n⚠️  场景 B 比 eager **慢 {fb.cost()[0]/fb.cost()[1]:.1f} 倍** —— 编译成了纯亏损。')
print('✅ 场景 D 是个隐蔽的坑：把 scale 包成 tensor（tf.constant / torch.tensor）就能避免。')

In [ ]:
# input_signature：把可变维标成 None，一张图搞定所有形状
fe = Function(model, input_signature=[((None, None), 'float32'), ((None, 8), 'float32')])
for i in range(500):
    fe([((32, 8 + i % 200), 'float32'), ((8 + i % 200, 8), 'float32')])
print(f'声明了 input_signature（可变维=None）: 调用 {fe.n_calls}, 追踪 {fe.n_traces}')
assert fe.n_traces == 1, '声明动态维后只追踪一次'
tg, te = fe.cost()
print(f'相对 eager: {tg/te:.2f}×  （从 {fb.cost()[0]/fb.cost()[1]:.1f}× 变成 {tg/te:.2f}×）')
print('\n✅ 代价：失去基于静态形状的部分优化（如常量折叠、精确的内存规划）。')
print('   真实对应: tf.function(input_signature=[tf.TensorSpec([None, None], tf.float32)])')
print('             torch.compile(model, dynamic=True)')
print('             torch.onnx.export(..., dynamic_axes={...})   ← 模块 03')

### 重追踪的通用诊断技巧（跨框架都能用）

**在被编译的函数里放一个纯 Python 副作用，数它执行了几次。执行次数 = 追踪次数。**

In [ ]:
def make_traced_with_counter(fn):
    counter = {'n': 0}
    def wrapped(*args, **kw):
        counter['n'] += 1          # ← 纯 Python，只在追踪时执行
        return fn(*args, **kw)
    return wrapped, counter

wrapped, cnt = make_traced_with_counter(model)
ff = Function(wrapped)
for i in range(300):
    ff([((32, 8 + i % 5), 'float32'), ((8 + i % 5, 8), 'float32')])
print(f'调用 300 次，Python 函数体执行了 {cnt["n"]} 次 -> 追踪了 {cnt["n"]} 次')
assert cnt['n'] == ff.n_traces == 5
print('\n✅ 这个技巧跨框架通用：')
print('   TF     : 在 @tf.function 里放 print("tracing")，数它出现几次')
print('   PyTorch: TORCH_LOGS="recompiles" 或在函数里放 print')
print('   JAX    : 在 jit 的函数里放 print（tracer 期才执行）')
print('   **每步训练都打印 = 你抓到了重追踪。**')

## 4 · 框架默认值对照：三个「不报错但数值错」的坑

这三个是跨框架迁移事故的最高发区（模块 02 会做成完整的映射表）。

In [ ]:
def conv_weight_torch_to_tf(w_torch):
    '''PyTorch (out, in, kH, kW) -> TF (kH, kW, in, out)。**必须转置，不能 reshape**。'''
    return np.transpose(w_torch, (2, 3, 1, 0))

def conv_weight_tf_to_torch(w_tf):
    return np.transpose(w_tf, (3, 2, 0, 1))

w_t = rng.normal(size=(16, 3, 5, 5))            # out=16, in=3, k=5x5
w_tf = conv_weight_torch_to_tf(w_t)
print(f'PyTorch conv 权重 {w_t.shape} -> TF {w_tf.shape}')
assert w_tf.shape == (5, 5, 3, 16)
assert np.allclose(conv_weight_tf_to_torch(w_tf), w_t), '往返转换必须无损'
# 反面：直接 reshape（形状对了，内容全乱）
w_wrong = w_t.reshape(5, 5, 3, 16)
assert w_wrong.shape == w_tf.shape
assert not np.allclose(w_wrong, w_tf), '⚠️ reshape 的形状正确但内容完全不同！'
print('❌ 直接 reshape 得到的形状**一样**，但内容完全不同 —— 不报错、结果错。')
print(f'   两者最大差异: {np.abs(w_wrong - w_tf).max():.4f}')

In [ ]:
def layernorm(x, gamma, beta, eps):
    mu = x.mean(-1, keepdims=True)
    var = x.var(-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps) * gamma + beta

EPS_TORCH, EPS_KERAS = 1e-5, 1e-3
d = 16
gamma, beta = np.ones(d), np.zeros(d)
print(f"{'输入尺度':>10s} {'torch(1e-5)':>13s} {'keras(1e-3)':>13s} {'最大差异':>10s}")
for scale in [1.0, 1e-1, 1e-2, 1e-3]:
    x = rng.normal(size=(4, d)) * scale
    a = layernorm(x, gamma, beta, EPS_TORCH)
    b = layernorm(x, gamma, beta, EPS_KERAS)
    print(f'{scale:>10.0e} {np.abs(a).mean():>13.4f} {np.abs(b).mean():>13.4f} '
          f'{np.abs(a-b).max():>10.2e}')

x_small = rng.normal(size=(4, d)) * 1e-3
diff = np.abs(layernorm(x_small, gamma, beta, EPS_TORCH)
              - layernorm(x_small, gamma, beta, EPS_KERAS)).max()
assert diff > 1e-3, '激活值很小时，两个 eps 的差异变得可观'
print(f'\n⚠️  keras.layers.LayerNormalization 的默认 eps 是 **1e-3**，torch 是 **1e-5** ——')
print(f'    相差 100 倍。激活值小时最大差异达 {diff:.2e}，会在逐层对拍里暴露为「后面几层全错」。')
print('✅ 迁移时必须显式设 epsilon，不要用默认值。')

In [ ]:
# BatchNorm 的 momentum 语义**相反**
def bn_update_torch(running, batch_stat, momentum):
    '''PyTorch: running = (1 - m) * running + m * batch    （m 是**新值**的权重）'''
    return (1 - momentum) * running + momentum * batch_stat

def bn_update_tf(moving, batch_stat, momentum):
    '''TF/Keras: moving = m * moving + (1 - m) * batch     （m 是**旧值**的权重）'''
    return momentum * moving + (1 - momentum) * batch_stat

r_t = r_f = 0.0
batch = 1.0
for _ in range(10):
    r_t = bn_update_torch(r_t, batch, momentum=0.1)     # torch 默认
    r_f = bn_update_tf(r_f, batch, momentum=0.99)       # keras 默认
print(f'10 步后: torch(m=0.1) running={r_t:.4f} | keras(m=0.99) moving={r_f:.4f}')
# 对应关系：torch 的 m 对应 keras 的 1-m
r_a = r_b = 0.0
for _ in range(10):
    r_a = bn_update_torch(r_a, batch, 0.1)
    r_b = bn_update_tf(r_b, batch, 0.9)                  # 1 - 0.1
assert abs(r_a - r_b) < 1e-12, 'torch 的 momentum=m 对应 keras 的 momentum=1-m'
print(f'✅ 对应关系: torch momentum=0.1  ⟺  keras momentum=0.9  (两者都得 {r_a:.4f})')
print(f'⚠️  照抄数字（两边都写 0.1 或都写 0.99）会让滑动统计的更新速度差 9 倍 ——')
print('    训练时看不出来，**切到 eval 模式后才暴露**（因为 eval 才用滑动统计）。')

## 5 · 编译的盈亏平衡

In [ ]:
def breakeven_calls(n_traces, trace_cost, t_eager, t_graph):
    '''值得编译 <=> N > n_traces * trace_cost / (t_eager - t_graph)'''
    if t_graph >= t_eager:
        return float('inf')
    return n_traces * trace_cost / (t_eager - t_graph)

TRACE_COST = 200.0
print(f"{'场景':<22s} {'追踪次数':>8s} {'加速比':>7s} {'盈亏平衡调用数':>15s}")
rows = [('形状固定的训练', 1, 1.3), ('形状固定的推理', 1, 1.5),
        ('形状分 8 桶', 8, 1.3), ('形状每次都变(500)', 500, 1.3)]
for name, nt, speed in rows:
    be = breakeven_calls(nt, TRACE_COST, speed, 1.0)
    print(f'{name:<22s} {nt:>8d} {speed:>6.1f}× {be:>15,.0f}')

be_fixed = breakeven_calls(1, TRACE_COST, 1.3, 1.0)
be_bucket = breakeven_calls(8, TRACE_COST, 1.3, 1.0)
be_var = breakeven_calls(500, TRACE_COST, 1.3, 1.0)
assert be_fixed < 1000 and be_bucket < 10000
assert be_var > 300_000, '形状每次都变时基本不可能回本'
assert breakeven_calls(1, TRACE_COST, 1.0, 1.0) == float('inf'), '没有加速就永远不回本'
print(f'\n✅ 形状固定时 {be_fixed:.0f} 次调用就回本（训练几分钟就够）；')
print(f'   形状每次都变时要 {be_var:,.0f} 次 —— 实际上永远不回本。')
print('   三条解法：形状分桶 / 声明动态维 / 只编译形状稳定的热点部分。')

## ✏️ 练习 1：签名缓存的 key

实现 `signature_key(input_specs, python_args, dynamic_dims=())`：
返回一个可哈希的 key。`dynamic_dims` 是被声明为动态的维度下标集合（如 `(0, 1)` 表示
每个输入的第 0、1 维都不参与 key）。被声明为动态的维度在 key 里用 `None` 代替。

In [ ]:
def signature_key(input_specs, python_args, dynamic_dims=()):
    # TODO: 对每个 (shape, dtype)，把 dynamic_dims 里的维度换成 None；
    #       返回 (tuple(处理后的 specs), tuple(sorted(python_args.items())))
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
s1 = [((32, 128), 'float32'), ((128, 8), 'float32')]
s2 = [((32, 256), 'float32'), ((256, 8), 'float32')]
assert signature_key(s1, {}) != signature_key(s2, {}), '不声明动态维 -> 不同形状不同 key'
assert signature_key(s1, {}, dynamic_dims=(1,)) != signature_key(s2, {}, dynamic_dims=(1,)) \
       or True     # 第二个输入的第 0 维仍不同
assert signature_key(s1, {}, dynamic_dims=(0, 1)) == signature_key(s2, {}, dynamic_dims=(0, 1)), \
    '把所有可变维声明为动态 -> 同一个 key -> 只追踪一次'
assert signature_key(s1, {'scale': 1.0}) != signature_key(s1, {'scale': 2.0}), \
    'Python 参数参与 key'
assert signature_key(s1, {'a': 1, 'b': 2}) == signature_key(s1, {'b': 2, 'a': 1}), \
    'Python 参数的顺序不应影响 key'
print('key(s1)                =', signature_key(s1, {}))
print('key(s1, dynamic=(0,1)) =', signature_key(s1, {}, (0, 1)))
print('✅ 练习 1 通过：这就是 tf.function 的 input_signature / torch.compile 的 dynamic')

## ✏️ 练习 2：框架默认值映射

实现 `translate_defaults(framework_from, framework_to, layer, params)`：
把一个框架的层参数翻译成另一个框架的。至少支持：
- `('torch','keras','LayerNorm', {'eps': 1e-5})` → `{'epsilon': 1e-5}`（**值不变，键名变**）
- `('torch','keras','BatchNorm', {'momentum': 0.1})` → `{'momentum': 0.9}`（**值要取 1-m**）
- 反方向同理。未知层抛 `ValueError`。

In [ ]:
def translate_defaults(framework_from, framework_to, layer, params):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert translate_defaults('torch', 'keras', 'LayerNorm', {'eps': 1e-5}) == {'epsilon': 1e-5}
assert translate_defaults('keras', 'torch', 'LayerNorm', {'epsilon': 1e-3}) == {'eps': 1e-3}
assert translate_defaults('torch', 'keras', 'BatchNorm', {'momentum': 0.1}) == {'momentum': 0.9}
assert translate_defaults('keras', 'torch', 'BatchNorm', {'momentum': 0.99}) == \
       {'momentum': 0.01} or abs(translate_defaults('keras','torch','BatchNorm',
                                 {'momentum': 0.99})['momentum'] - 0.01) < 1e-12
try:
    translate_defaults('torch', 'keras', 'Mystery', {}); raise RuntimeError('不该到这')
except ValueError as e:
    print(f'未知层报错: {e}')
print('LayerNorm torch->keras:', translate_defaults('torch','keras','LayerNorm',{'eps':1e-5}))
print('BatchNorm torch->keras:', translate_defaults('torch','keras','BatchNorm',{'momentum':0.1}))
print('✅ 练习 2 通过：**键名变**与**值要换算**是两类不同的坑，都不报错')

## ✏️ 练习 3：该不该编译

实现 `should_compile(n_calls, n_distinct_shapes, trace_cost, speedup)`：
返回 `(是否值得, 编译总成本, eager 总成本)`。假设 eager 单次成本为 1.0，
图执行单次成本为 `1/speedup`，追踪次数 = `min(n_distinct_shapes, n_calls)`。

In [ ]:
def should_compile(n_calls, n_distinct_shapes, trace_cost=200.0, speedup=1.3):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
ok, cg, ce = should_compile(100_000, 1)
assert ok, f'固定形状 + 10 万次调用应该值得编译 ({cg:.0f} vs {ce:.0f})'
ok2, cg2, ce2 = should_compile(300, 300)
assert not ok2, '每次形状都变 + 只调 300 次 -> 不值得'
ok3, *_ = should_compile(100_000, 8)
assert ok3, '分 8 桶 + 10 万次调用 -> 值得'
ok4, *_ = should_compile(50, 1)
assert not ok4, '只调 50 次 -> 追踪成本都摊不回来'
for args in [(100_000, 1), (100_000, 8), (300, 300), (50, 1)]:
    o, a, b = should_compile(*args)
    print(f'调用 {args[0]:>6d}, 不同形状 {args[1]:>4d} -> '
          f'编译 {a:>10.0f} vs eager {b:>8.0f}  {"✅ 编译" if o else "❌ 别编译"}')
print('✅ 练习 3 通过：编译不是免费的，「形状稳定 + 调用次数多」才划算')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def signature_key(input_specs, python_args, dynamic_dims=()):
    dyn = set(dynamic_dims)
    specs = tuple((tuple(None if i in dyn else d for i, d in enumerate(shape)), dtype)
                  for shape, dtype in input_specs)
    return (specs, tuple(sorted(python_args.items())))

In [ ]:
# 练习 2 参考答案
def translate_defaults(framework_from, framework_to, layer, params):
    if layer == 'LayerNorm':
        if (framework_from, framework_to) == ('torch', 'keras'):
            return {'epsilon': params['eps']}
        if (framework_from, framework_to) == ('keras', 'torch'):
            return {'eps': params['epsilon']}
    elif layer == 'BatchNorm':
        # 语义相反：torch 的 momentum 是**新值**权重，keras 是**旧值**权重
        return {'momentum': 1.0 - params['momentum']}
    raise ValueError(f'unknown layer {layer!r} for {framework_from}->{framework_to}')

In [ ]:
# 练习 3 参考答案
def should_compile(n_calls, n_distinct_shapes, trace_cost=200.0, speedup=1.3):
    n_traces = min(n_distinct_shapes, n_calls)
    cost_graph = n_traces * trace_cost + n_calls * (1.0 / speedup)
    cost_eager = n_calls * 1.0
    return cost_graph < cost_eager, cost_graph, cost_eager

---
## 🧪 真实 API 对照胶囊（不在本环境运行，可原样复制）

In [ ]:
RECIPE = r'''
# ── TensorFlow：tf.function 的正确用法 ──────────────────────────────
import tensorflow as tf

# ① 声明输入签名 -> 可变维用 None -> 只追踪一次
@tf.function(input_signature=[
    tf.TensorSpec(shape=[None, None], dtype=tf.int32, name="input_ids"),
    tf.TensorSpec(shape=[None, None], dtype=tf.int32, name="attention_mask"),
])
def serve(input_ids, attention_mask):
    # ② 调试用 tf.print（图内算子，每次执行都打印），不要用 Python print
    tf.print("batch:", tf.shape(input_ids)[0])
    logits = model(input_ids, attention_mask=attention_mask, training=False)
    # ③ 依赖 tensor 值的分支必须用图算子
    return tf.cond(tf.reduce_max(logits) > 10.0,
                   lambda: tf.nn.softmax(logits / 2.0),
                   lambda: tf.nn.softmax(logits))

# ④ 诊断重追踪：数 Python 侧的执行次数
tf.config.run_functions_eagerly(False)
# 或设环境变量 TF_FUNCTION_JIT_COMPILE_DEFAULT / 看 "retracing" 警告

# ⑤ 显存：TF 默认会占满整卡，几乎总是要关掉
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

# ── PyTorch：同样的语义，不同的名字 ─────────────────────────────────
import torch
compiled = torch.compile(model, dynamic=True)     # 对应 input_signature 的 None
# 诊断重编译：
#   TORCH_LOGS="recompiles,graph_breaks" python train.py
#   torch._dynamo.config.cache_size_limit = 64

# ── JAX：追踪语义最显式 ────────────────────────────────────────────
import jax
@jax.jit
def f(params, x):
    # 对 tracer 做 Python 判断会直接抛 ConcretizationTypeError（**宁可报错也不静默固化**）
    return jax.lax.cond(x.sum() > 0, lambda: x * 2, lambda: x)
'''
print(RECIPE)
for c in ['input_signature', 'tf.print', 'tf.cond', 'set_memory_growth',
          'dynamic=True', 'recompiles', 'lax.cond']:
    assert c in RECIPE, c
print('✅ 配方覆盖本模块全部要点（签名/图内打印/图内控制流/显存/重追踪诊断/三框架对照）')

### 小结
- 静态图 = **Python 只是构图脚本**。这一句能解释 80% 的诡异行为：print 只出现一次、计数器不动、控制流被固化。
- **区分「构图时的量」与「运行时的量」**：依赖运行时值的控制流必须用图算子（`tf.cond`/`lax.cond`/`torch.cond`）。
- **重追踪是头号性能杀手**：形状变、dtype 变、传 Python 标量、在循环里新建函数对象，都会让缓存失效。
  诊断技巧跨框架通用：**在函数里放一个 Python 副作用，数它执行几次**。
- **形状每次都变时，编译是纯亏损**（本模块算出比 eager 还慢）。三条解法：分桶 / 声明动态维 / 只编译热点。
- **Keras 三种 API**：Functional 构建的是「可检查的数据结构」（能 summary、能直接导出）；Subclassing 是纯 Python（最像 PyTorch，但失去静态检查）。
- **三个「不报错但数值错」的坑**：Conv 权重布局（必须转置不能 reshape）、LayerNorm 的 eps（1e-5 vs 1e-3）、BatchNorm 的 momentum（**语义相反**，0.1 ↔ 0.9）。
- 三框架的差别在于**状态放哪**与**图从哪来**：PyTorch 状态在对象里+事后追踪；TF 状态在层里+装饰器；JAX 无隐藏状态+可组合变换。

下一站：**模块 02 · 框架迁移与权重对齐** —— 把权重搬过去，并**证明**它真的等价。